# Week 7: AutoML and Transfer Learning

This notebook demonstrates:
1. Cross-validation techniques
2. Bias-variance tradeoff visualization
3. AutoML with AutoGluon
4. Transfer learning for text (HuggingFace)
5. Transfer learning for vision (timm)

In [ ]:
# Install dependencies (run once)
# !pip install scikit-learn pandas numpy matplotlib seaborn
# !pip install autogluon
# !pip install transformers torch
# !pip install timm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, 
    GridSearchCV, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

plt.style.use('seaborn-v0_8-whitegrid')
%config InlineBackend.figure_format = 'retina'

## Part 1: Cross-Validation

In [ ]:
# Generate sample data
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)

print(f"Data shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# Problem: Single train/test split varies with random seed
accuracies = []

for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    accuracies.append(model.score(X_test, y_test))

print(f"Accuracy varies from {min(accuracies):.1%} to {max(accuracies):.1%}")
print(f"Range: {(max(accuracies) - min(accuracies))*100:.1f} percentage points!")

In [ ]:
# Solution: K-Fold Cross-Validation
model = RandomForestClassifier(n_estimators=50, random_state=42)

# 5-fold CV
scores = cross_val_score(model, X, y, cv=5)

print("5-Fold Cross-Validation:")
print(f"  Fold scores: {scores}")
print(f"  Mean: {scores.mean():.3f}")
print(f"  Std:  {scores.std():.3f}")
print(f"  Report as: {scores.mean():.1%} ± {scores.std():.1%}")

In [ ]:
# Stratified K-Fold for imbalanced data
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    fold_scores.append(score)
    
    # Check class distribution is preserved
    train_ratio = y_train.mean()
    test_ratio = y_test.mean()
    print(f"Fold {fold}: Train class ratio={train_ratio:.2f}, Test class ratio={test_ratio:.2f}, Score={score:.3f}")

print(f"\nMean: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f}")

## Part 2: Bias-Variance Tradeoff

In [ ]:
# Visualize overfitting with Decision Tree depth
from sklearn.model_selection import validation_curve

depths = range(1, 20)
train_scores, test_scores = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X, y,
    param_name="max_depth",
    param_range=depths,
    cv=5,
    scoring="accuracy"
)

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores.mean(axis=1), 'b-', label='Training score', linewidth=2)
plt.plot(depths, test_scores.mean(axis=1), 'r-', label='Validation score', linewidth=2)
plt.fill_between(depths, 
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1),
                 alpha=0.1, color='b')
plt.fill_between(depths,
                 test_scores.mean(axis=1) - test_scores.std(axis=1),
                 test_scores.mean(axis=1) + test_scores.std(axis=1),
                 alpha=0.1, color='r')

plt.xlabel('Tree Depth (Model Complexity)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Bias-Variance Tradeoff: Decision Tree Depth', fontsize=14)
plt.legend(loc='best')
plt.axvline(x=test_scores.mean(axis=1).argmax()+1, color='g', linestyle='--', label='Optimal depth')
plt.tight_layout()
plt.show()

optimal_depth = test_scores.mean(axis=1).argmax() + 1
print(f"Optimal depth: {optimal_depth}")
print(f"At this depth: Train={train_scores.mean(axis=1)[optimal_depth-1]:.3f}, Val={test_scores.mean(axis=1)[optimal_depth-1]:.3f}")

In [ ]:
# Diagnosing your model
def diagnose_model(model, X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    gap = train_acc - test_acc
    
    if train_acc < 0.7:
        diagnosis = "UNDERFITTING (high bias)"
        fix = "Try more complex model or more features"
    elif gap > 0.1:
        diagnosis = "OVERFITTING (high variance)"
        fix = "Try regularization, simpler model, or more data"
    else:
        diagnosis = "GOOD FIT"
        fix = "Model is well-calibrated"
    
    print(f"Train accuracy: {train_acc:.1%}")
    print(f"Test accuracy:  {test_acc:.1%}")
    print(f"Gap: {gap:.1%}")
    print(f"Diagnosis: {diagnosis}")
    print(f"Suggestion: {fix}")
    return train_acc, test_acc

print("=" * 50)
print("Shallow tree (likely underfitting):")
diagnose_model(DecisionTreeClassifier(max_depth=2), X, y)

print("\n" + "=" * 50)
print("Deep tree (likely overfitting):")
diagnose_model(DecisionTreeClassifier(max_depth=None), X, y)

print("\n" + "=" * 50)
print("Random Forest (usually good fit):")
diagnose_model(RandomForestClassifier(n_estimators=100, max_depth=10), X, y)

## Part 3: Baseline Models Comparison

In [ ]:
# Compare different baseline models
models = {
    'Dummy (majority)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10),
}

results = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)
    results.append({
        'Model': name,
        'Mean': scores.mean(),
        'Std': scores.std(),
        'Scores': scores
    })
    print(f"{name:25s}: {scores.mean():.3f} ± {scores.std():.3f}")

# Visualize
plt.figure(figsize=(10, 6))
for i, r in enumerate(results):
    plt.scatter([i] * 5, r['Scores'], alpha=0.6, s=100)
    plt.scatter(i, r['Mean'], color='red', s=200, marker='x', linewidths=3)

plt.xticks(range(len(results)), [r['Model'] for r in results], rotation=15)
plt.ylabel('Accuracy')
plt.title('Model Comparison with 5-Fold CV')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]  # Top 10

plt.figure(figsize=(10, 6))
plt.bar(range(10), importances[indices])
plt.xticks(range(10), [f'Feature {i}' for i in indices])
plt.ylabel('Importance')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

## Part 4: AutoML with AutoGluon

In [ ]:
# Create a sample dataset
from sklearn.datasets import fetch_california_housing

# Use Iris for classification demo
iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df['target'] = iris.target

print("Iris dataset:")
print(iris_df.head())
print(f"\nShape: {iris_df.shape}")

In [ ]:
# AutoGluon demo (uncomment to run - takes a few minutes)
# from autogluon.tabular import TabularPredictor

# # Split data
# train_df = iris_df.sample(frac=0.8, random_state=42)
# test_df = iris_df.drop(train_df.index)

# # Train with AutoGluon
# predictor = TabularPredictor(label='target', path='autogluon_models')
# predictor.fit(train_df, time_limit=60)  # 1 minute

# # Show leaderboard
# print(predictor.leaderboard(test_df))

In [ ]:
# Simulated AutoGluon-like comparison
print("Simulated AutoML Leaderboard:")
print("(In practice, use AutoGluon for this)")
print()

from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM': SVC(),
}

leaderboard = []
for name, model in models.items():
    scores = cross_val_score(model, iris.data, iris.target, cv=5)
    leaderboard.append({'Model': name, 'Score': scores.mean(), 'Std': scores.std()})

leaderboard_df = pd.DataFrame(leaderboard).sort_values('Score', ascending=False)
print(leaderboard_df.to_string(index=False))

## Part 5: Transfer Learning for Text

In [ ]:
# Text classification with HuggingFace transformers
from transformers import pipeline

# Load pretrained sentiment analysis model
classifier = pipeline("sentiment-analysis")

# Test on movie reviews
reviews = [
    "This movie was absolutely fantastic! Best film of the year.",
    "Terrible waste of time. I want my money back.",
    "It was okay, nothing special but watchable.",
    "The acting was superb and the plot kept me engaged throughout.",
    "Boring, predictable, and way too long."
]

print("Sentiment Analysis (Zero Training Required!)")
print("=" * 60)
for review in reviews:
    result = classifier(review)[0]
    print(f"{review[:50]:50s} → {result['label']:8s} ({result['score']:.2f})")

In [ ]:
# Zero-shot classification - no training needed!
zero_shot = pipeline("zero-shot-classification")

text = "The new iPhone 15 has an amazing camera and great battery life."
labels = ["technology", "sports", "politics", "entertainment", "science"]

result = zero_shot(text, labels)

print(f"Text: {text}")
print(f"\nPredicted labels:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:15s}: {score:.3f}")

In [ ]:
# Try your own text!
your_text = "The weather today is sunny and perfect for a picnic."
your_labels = ["weather", "food", "sports", "travel"]

result = zero_shot(your_text, your_labels)
print(f"Text: {your_text}")
print(f"Top prediction: {result['labels'][0]} ({result['scores'][0]:.2f})")

## Part 6: Transfer Learning for Vision

In [ ]:
# Vision transfer learning with timm
import timm

# List available models
print("Popular pretrained models in timm:")
popular_models = ['resnet50', 'efficientnet_b0', 'vit_base_patch16_224', 'convnext_base']
for model_name in popular_models:
    print(f"  - {model_name}")

print(f"\nTotal models available: {len(timm.list_models())}")

In [ ]:
# Load a pretrained model
import torch

# Create model with custom number of classes
model = timm.create_model('resnet18', pretrained=True, num_classes=5)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: ResNet-18")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Feature extraction: Freeze all layers except the head
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the final classification layer
for param in model.fc.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"After freezing:")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Only training {trainable_params/total_params:.1%} of the model!")

## Summary

**Key takeaways:**

1. **Cross-validation**: Always use K-fold CV instead of single train/test split
2. **Bias-variance tradeoff**: Watch train vs test accuracy gap
3. **Baseline models**: Start simple (LogReg, RF) before complex
4. **AutoML**: Use AutoGluon for tabular data
5. **Transfer learning**: Use pretrained models for text (HuggingFace) and images (timm)